---------------------------------------------------------------------------------------------------------------
# Visualization Generation and Saving
---------------------------------------------------------------------------------------------------------------
## Content Log:
1. Import Libraries
2. Setting Styling for Visuals
3. Load Data
4. Creating FOLDERS to save the Visuals
5. Data PArsing for "Date" Column
6. Creating and Saving Heatmaps
7. Creating and saving Price distribution Histograms
8. Creating and Saving line plots for Price over time ("Date")
9. Creating and saving line plots for Volume over time ("Date")
10. Creating and saving Correlation Heatmaps
11. Creating and saving Daily_Return distribution plots
12. Creating and saving Rolling_Volatility graphs for 30 days
13. Creating and saving Boxplots
14. Combining all Datasets for overall Heatmap
15. Creating and saving Overall Heatmap
---------------------------------------------------------------------------------------------------------------

# 01. Import Libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# 02. Setting Stylings for the Visuals

In [6]:
plt.rcParams["figure.figsize"] = (12,6)
sns.set_style("whitegrid")

# 03. Load Data

In [7]:
data_path = "../../data/raw_data"
datasets = {}
for file in os.listdir(data_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(data_path, file))
        df["Ticker"] = file.replace(".csv","")
        datasets[file] = df

# 04. Creating FOLDERS to save the Visuals

In [8]:
base_dir = "../../visuals_and_reports/eda_plots"
os.makedirs(base_dir, exist_ok=True)
folders = [
    "heatmaps",
    "price_distribution_histograms",
    "price_over_time_lineplots",
    "volume_over_time_lineplots",
    "correlation_heatmaps",
    "daily_return_distribution",
    "rolling_volatility_30d",
    "boxplots"
]
for f in folders:
    os.makedirs(os.path.join(base_dir, f), exist_ok=True)

# 05. Data Parsing for "Date" column

In [9]:
for name, df in datasets.items():
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"])
        df.sort_values("Date", inplace=True)
        df.reset_index(drop=True, inplace=True)

# 06. Creating and saving Heatmaps

In [6]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    plt.figure()
    sns.heatmap(df.isnull(), cbar=False)
    plt.title(f"Missing Values Heatmap: {ticker}")
    save_path = f"../../visuals_and_reports/eda_plots/heatmaps/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 07. Creating and saving Price distribution Histograms

In [7]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    plt.figure()
    sns.histplot(df["Close"], bins=50, kde=True)
    plt.title(f"Close Price Distribution: {ticker}")
    plt.xlabel("Close Price")
    plt.ylabel("Frequency")
    save_path = f"../../visuals_and_reports/eda_plots/price_distribution_histograms/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 08. Creating and saving line plots for Price over time ("Date")

In [8]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    plt.figure()
    plt.plot(df["Date"], df["Close"])
    plt.title(f"Closing Price Over Time: {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Close Price")
    save_path = f"../../visuals_and_reports/eda_plots/price_over_time_lineplots/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 09. Creating and saving line plots for Volume over time ("Date")

In [9]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    plt.figure()
    plt.plot(df["Date"], df["Volume"])
    plt.title(f"Volume Over Time: {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Volume")
    save_path = f"../../visuals_and_reports/eda_plots/volume_over_time_lineplots/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 10. Creating and saving Correlation Heatmaps

In [10]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    numeric_df = df.select_dtypes(include=[np.number])
    plt.figure()
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
    plt.title(f"Correlation Matrix: {ticker}")
    save_path = f"../../visuals_and_reports/eda_plots/correlation_heatmaps/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 11. Creating and saving Daily_Return distribution plots

In [11]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    df["Daily_Return"] = df["Close"].pct_change()
    plt.figure()
    sns.histplot(df["Daily_Return"].dropna(), bins=50, kde=True)
    plt.title(f"Daily Return Distribution: {ticker}")
    plt.xlabel("Daily Return")
    plt.ylabel("Frequency")
    save_path = f"../../visuals_and_reports/eda_plots/daily_return_distribution/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 12. Creating and saving Rolling_Volatility graphs for 30 days

In [12]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    df["Rolling_Volatility"] = df["Daily_Return"].rolling(window=30).std()
    plt.figure()
    plt.plot(df["Date"], df["Rolling_Volatility"])
    plt.title(f"30 Day Rolling Volatility: {ticker}")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    save_path = f"../../visuals_and_reports/eda_plots/rolling_volatility_30d/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 13. Creating and saving Boxplots

In [13]:
for name, df in datasets.items():
    ticker = name.replace(".csv","")
    plt.figure()
    sns.boxplot(data=df[["Open","High","Low","Close"]])
    plt.title(f"Price Outlier Detection: {ticker}")
    save_path = f"../../visuals_and_reports/eda_plots/boxplots/{ticker}.png"
    plt.savefig(save_path)
    plt.close()

# 14. Combining all Datasets for overall Heatmap

In [10]:
combined_df = pd.concat(datasets.values(), ignore_index=True)

# 15. Creating and saving Overall Heatmap

In [15]:
pivot_close = combined_df.pivot_table(
    values="Close",
    index="Date",
    columns="Ticker"
)
plt.figure(figsize=(20,24))
sns.heatmap(pivot_close.corr(), cmap="coolwarm", annot=True)
plt.title("Cross Stock Correlation Nifty 50 Universe")
save_path = "../../visuals_and_reports/eda_plots/cross_stock_correlation_heatmap.png"
plt.savefig(save_path)
plt.close()